# Demo: proposed method (HGT)
Creates the model, loads trained weights, runs inference on one demo region and plots the predicted land-cover maps.

Configuration of the trained model: **HGT** (`MyModelGraphTransformer`), **0 spatial steps** (spatial region 0), **1 temporal step into the past** with a temporal neighborhood (fanout) of **2**.

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import json
import numpy as np
import torch
import dgl
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

sys.path.append("../source")
from proposed_method_lib.graph import load_item_with_pickle
from proposed_method_lib.dgl import MyModelGraphTransformer, ProposedSampler

## Paths

In [ ]:
WEIGHTS_PATH = "./proposed_method_weights/best_model.pt"
NORMALIZATION_PATH = "./source/train_set_normalization_data.json"
DEMO_REGION_PATH = None #Must set

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Model / sampling parameters
These correspond to the `run_training.py` arguments used for the best model:

| Training argument | Value |
|---|---|
| `--use_transformer` | set (HGT) |
| `--spatial_step_surrounding_region` | 0 |
| `--time_step_past` | 1 |
| `--time_neighborhood_size` | 2 |
| `--use_custom_sampler` | set (`ProposedSampler`) |
| `--h_feat_amount` | 256 |
| `--num_heads` | 8 |
| `--feature_extraction_NN` | EfficientNetV2-S |
| `--region_bbox_size` / `--region_bbox_channels` | 32 / 4 (RGB + NIR) |

In [ ]:
SPATIAL_STEPS = 0           # --spatial_step_surrounding_region
SPACE_NEIGHBORHOOD_SIZE = 0  # --space_neighborhood_size (unused, since SPATIAL_STEPS = 0)
TIME_STEPS_PAST = 1         # --time_step_past
TIME_NEIGHBORHOOD_SIZE = 2  # --time_neighborhood_size

H_FEAT_AMOUNT = 256
NUM_HEADS = 8
FEATURE_EXTRACTOR = "EfficientNetV2-S"
BBOX_SIZE = 32
BBOX_CHANNELS = 4
BATCH_SIZE = 60

# Same logic as in run_training.py
sampling_amount = [SPACE_NEIGHBORHOOD_SIZE] * SPATIAL_STEPS + [TIME_NEIGHBORHOOD_SIZE] * TIME_STEPS_PAST
if TIME_STEPS_PAST == 0 and SPATIAL_STEPS == 0:
    sampling_amount = [0, 0]
elif TIME_STEPS_PAST == 0 and SPATIAL_STEPS > 0:
    sampling_amount = sampling_amount + [0]
elif TIME_STEPS_PAST > 0 and SPATIAL_STEPS == 0:
    sampling_amount = [0] + sampling_amount

if SPATIAL_STEPS == 0 and TIME_STEPS_PAST > 0:
    h_feats_count = TIME_STEPS_PAST
elif SPATIAL_STEPS > 0 and TIME_STEPS_PAST == 0:
    h_feats_count = SPATIAL_STEPS
elif SPATIAL_STEPS == 0 and TIME_STEPS_PAST == 0:
    h_feats_count = 1
else:
    h_feats_count = SPATIAL_STEPS + TIME_STEPS_PAST - 1
h_feats = [H_FEAT_AMOUNT] * h_feats_count

edge_type_conversion = {'temporal': 0, 'spatial': 1}
edge_sequence_stages = [(edge_type_conversion['spatial'], max(SPATIAL_STEPS, 1)),
                        (edge_type_conversion['temporal'], max(TIME_STEPS_PAST, 1))]

print("sampling_amount:", sampling_amount)  # -> [0, 2]
print("h_feats:", h_feats)                  # -> [256]
print("edge_sequence_stages:", edge_sequence_stages)

## Create model and load weights

In [ ]:
with open(NORMALIZATION_PATH) as f:
    norm = json.load(f)
train_channel_means = np.array([float(norm["means"][k]) for k in ["R", "G", "B", "infra"]])
train_channel_stds = np.array([float(norm["stds"][k]) for k in ["R", "G", "B", "infra"]])

state_dict = torch.load(WEIGHTS_PATH, map_location="cpu")

# number of classes = output size of the last HGT layer
last_conv = len(h_feats)
num_classes = [v for k, v in state_dict.items() if k.startswith(f"convs.{last_conv}.linear_q")][0].shape[-1]
print("num_classes:", num_classes)

model = MyModelGraphTransformer(h_feats=h_feats,
                                num_classes=num_classes,
                                image_size=(BBOX_SIZE, BBOX_SIZE, BBOX_CHANNELS),
                                feature_extractor_size=FEATURE_EXTRACTOR,
                                num_heads=NUM_HEADS,
                                feat_dropout=0.5,
                                use_edge_weights=False,
                                freeze_upper_N_layers=None,  # None -> skip ImageNet download, weights are loaded below
                                use_ngnn_modules=False,
                                max_freeze_percentage=80.0,
                                train_channel_means=train_channel_means,
                                train_channel_stds=train_channel_stds,
                                use_normalization=False,
                                aggregate_node_classifications=False)

model.load_state_dict(state_dict, strict=True)
model = model.to(device)
model.eval()

## Load demo data and build DGL graph

In [ ]:
graph_nx = load_item_with_pickle(DEMO_REGION_PATH + "graph.pickle")
segmentation_masks = load_item_with_pickle(DEMO_REGION_PATH + "segmentation_masks.pickle")  # (T, H, W)
print("Nodes:", graph_nx.number_of_nodes(), "/ edges:", graph_nx.number_of_edges(), "/ masks:", segmentation_masks.shape)

for u, v, data in graph_nx.edges(data=True):
    if isinstance(data['type'], str):
        data['type'] = edge_type_conversion[data['type']]

dgl_graph = dgl.from_networkx(graph_nx, node_attrs=["features", "label", "time", "index"], edge_attrs=["type"])
dgl_graph = dgl_graph.int()
dgl_graph.ndata['label'] = dgl_graph.ndata['label'].int()
dgl_graph.ndata['time'] = dgl_graph.ndata['time'].int()

assert dgl_graph.ndata['features'].shape[1] == BBOX_SIZE * BBOX_SIZE * BBOX_CHANNELS, "features do not match bbox size/channels"
del graph_nx

## Run inference

In [ ]:
node_sampler = ProposedSampler(fanouts=sampling_amount, edge_sequence_stages=edge_sequence_stages, gat_nn=False, edge_dir="in")

use_uva = device.type == "cuda"
node_indices = torch.arange(dgl_graph.number_of_nodes(), dtype=torch.int32)
if use_uva:
    node_indices = node_indices.to(device)

dataloader = dgl.dataloading.DataLoader(dgl_graph, node_indices, node_sampler,
                                        device=device, batch_size=BATCH_SIZE, shuffle=False,
                                        drop_last=False, num_workers=0, use_uva=use_uva)

node_predictions = np.zeros(dgl_graph.number_of_nodes(), dtype=np.int64)
with torch.no_grad():
    for i, (input_nodes, output_nodes, mfgs) in enumerate(dataloader):
        inputs = mfgs[0].srcdata['features'].float()
        logits = model(mfgs, inputs)
        node_predictions[output_nodes.cpu().numpy()] = logits.argmax(1).cpu().numpy()
        if i % 50 == 0:
            print(f"Batch {i + 1}/{len(dataloader)}")

## Map node predictions back to pixels

In [ ]:
node_time = dgl_graph.ndata['time'].numpy()
node_segment = dgl_graph.ndata['index'].numpy()
node_label = dgl_graph.ndata['label'].numpy()

NO_NODE = 255
pred_maps = np.full(segmentation_masks.shape, NO_NODE, dtype=np.uint8)
gt_maps = np.full(segmentation_masks.shape, NO_NODE, dtype=np.uint8)

for t in range(segmentation_masks.shape[0]):
    sel = node_time == t
    lut_pred = np.full(segmentation_masks[t].max() + 1, NO_NODE, dtype=np.uint8)
    lut_gt = lut_pred.copy()
    lut_pred[node_segment[sel]] = node_predictions[sel]
    lut_gt[node_segment[sel]] = node_label[sel]
    pred_maps[t] = lut_pred[segmentation_masks[t]]
    gt_maps[t] = lut_gt[segmentation_masks[t]]

print("Segment-level accuracy:", (node_predictions == node_label).mean())

## Plot

In [ ]:
class_names = ["Impervious surface", "Agriculture", "Forest & other vegetation", "Wetlands", "Soil", "Water", "Snow & ice"]
color_map = {0: (64, 64, 64), 1: (128, 128, 0), 2: (0, 128, 0), 3: (0, 0, 128),
             4: (139, 69, 19), 5: (0, 0, 255), 6: (192, 192, 192), NO_NODE: (0, 0, 0)}

def to_rgb(arr):
    rgb = np.zeros(arr.shape + (3,), dtype=np.uint8)
    for value, color in color_map.items():
        rgb[arr == value] = color
    return rgb

dates = ["2018_01", "2018_02", "2018_03", "2018_04", "2018_05", "2018_06", "2018_07", "2018_08", "2018_09", "2018_10", "2018_11", "2018_12",
         "2019_01", "2019_02", "2019_03", "2019_04", "2019_05", "2019_06", "2019_07", "2019_08", "2019_09", "2019_10", "2019_11", "2019_12"]

time_indices_to_plot = [0, 3, 6, 9]

fig, axes = plt.subplots(2, len(time_indices_to_plot), figsize=(5 * len(time_indices_to_plot), 10))
for col, t in enumerate(time_indices_to_plot):
    axes[0, col].imshow(to_rgb(gt_maps[t]))
    axes[0, col].set_title(f"Ground truth ({dates[t]})")
    axes[1, col].imshow(to_rgb(pred_maps[t]))
    axes[1, col].set_title(f"Prediction ({dates[t]})")
    axes[0, col].axis("off")
    axes[1, col].axis("off")

handles = [Patch(color=np.array(color_map[i]) / 255, label=name) for i, name in enumerate(class_names)]
fig.legend(handles=handles, loc="lower center", ncol=len(class_names))
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()